In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *
import pandas as pd

qt.qpa.plugin: Could not find the Qt platform plugin "wayland" in ""


In [67]:
files = sorted(glob.glob('/home/ulyanov/data/stereo/blos/*'))
#files = sorted(glob.glob('/home/ulyanov/data/cross/*'))
files

['/home/ulyanov/data/stereo/blos/hmi.M_45s.20250924_062015_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/stereo/blos/hmi.M_45s.20251007_124330_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/stereo/blos/solo_L2_phi-fdt-blos_20250924T061503_V202602220437_0549240502.fits.gz',
 '/home/ulyanov/data/stereo/blos/solo_L2_phi-fdt-blos_20251007T124003_V202602220437_0550070502.fits.gz']

In [68]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers.csv', skipinitialspace=True).drop(columns='date').dropna()
dids = df['did'].to_numpy()
xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']

In [69]:
file_hmi = files[0]
file_fdt = files[2]

with fits.open(file_hmi) as hdul:
    header_hmi = hdul[1].header
    data_hmi = hdul[1].data

with fits.open(file_fdt) as hdul:
    header_fdt = hdul[0].header
    data_fdt = hdul[0].data

did = int(file_fdt.split('_')[-1].split('.')[0])

data_fdt = undistort(data_fdt, header_fdt, xd, yd)
data_hmi = rebin(data_hmi, 8, update_header=header_hmi)

view_fdt = View.from_header(header_fdt).update(xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], rsun=ru_sun[dids == did][0],
                                               crota=header_fdt['CROTA'] + 0.25)
view_hmi = View.from_header(header_hmi)


view_new = view_fdt.update(crota=0)#, crlt=0)
transform_fdt = (view_new.to_carrington() -
                 view_fdt.to_carrington(mu_thr=0.2))


grid, alpha = transform_fdt(view_new.grid())
data_fdt = bilinear(data_fdt, *grid) * alpha

#view_new = view_fdt.update(crlt=view_hmi.crlt, crota=0)
transform_hmi = (view_new.to_carrington() -
                 view_hmi.to_carrington(mu_thr=0.2))

grid, alpha = transform_hmi(view_new.grid())
data_hmi = bilinear(data_hmi, *grid) * alpha


#data_fdt = rebin(data_fdt, 4)
#data_hmi = rebin(data_hmi, 4)


In [31]:
plt.figure(figsize=(10,10))
plt.imshow(data_hmi, cmap='seismic', vmin=-200, vmax=200)
plt.tight_layout()

In [32]:
plt.figure(figsize=(10,10))
plt.imshow(data_fdt, cmap='seismic', vmin=-200, vmax=200)
plt.tight_layout()

In [71]:
transform = view_hmi.to_carrington(origin='helioprojective') - view_fdt.to_carrington(origin='helioprojective')

e1 = (0,0,1)
e2 = transform(e1)[0]

e1 = np.array(e1)
e2 = np.array(e2)

q = np.sum(e1 * e2)


delta = np.sqrt(1 - q ** 2)

e2_ = (e2 - e1 * q) / delta

v1 = data_fdt.copy() * 1.2#
v2 = data_hmi.copy()

#v1[np.abs(v1) < 15] = np.nan
v2[np.abs(v2) < 10] = np.nan

v2_ = (v2 - v1 * q) / delta


print(e2_, np.arccos(q) * 180 / np.pi)

[0.99926351 0.03837227 0.        ] 37.10844172814751


In [77]:
xi, yi, zi = view_new.grid(origin='helioprojective')

plt.figure(figsize=(10,10))
plt.imshow(np.abs(v1 * zi + v2_ * (e2_[0] * xi + e2_[1] * yi)) / np.sqrt(v1 ** 2 + v2_ ** 2), 'jet', vmin=0, vmax=1)
plt.tight_layout()

In [83]:
xi, yi, zi = view_new.grid(origin='helioprojective')

plt.figure(figsize=(10,10))
plt.imshow(v1 * zi / np.sqrt(v1 ** 2 + v2_ ** 2), 'jet', vmin=-1, vmax=1)
plt.tight_layout()

In [36]:
files = sorted(glob.glob('/home/ulyanov/data/stereo/*'))
files

['/home/ulyanov/data/stereo/blos',
 '/home/ulyanov/data/stereo/solo_L2_phi-fdt-binc_20250924T061503_V202602220437_0549240502.fits.gz',
 '/home/ulyanov/data/stereo/vlos']

In [45]:
with fits.open(files[1]) as hdul:
    header = hdul[0].header
    data = hdul[0].data

data = undistort(data, header, xd, yd)

grid, alpha = transform_fdt(view_new.grid())
data = bilinear(data, *grid) * alpha

In [81]:
plt.figure(figsize=(10,10))
plt.imshow(np.cos(data * np.pi / 180), 'jet', vmin=-1, vmax=1)
plt.tight_layout()